Setup

In [ ]:
import os,json,glob, re, itertools
import numpy as np,pandas as pd
from sklearn.metrics import f1_score, classification_report
import warnings;
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

SRC= '/content/drive/MyDrive/Cyberbullying-Detection-in-Bilingual-English-Spanish-Text-A-Comparative-Study-/bilingual results'
OUT=f'{SRC}/extended_analysis'
os.makedirs(OUT, exist_ok=True)
CANDIDATES = [
    '/content/drive/MyDrive/Cyberbullying-Detection-in-Bilingual-English-Spanish-Text-A-Comparative-Study-/Datasets/Dataset_splits',
    '/content/drive/MyDrive/dissertation_datasets/Datasets/Dataset_splits',
]
SPLITS= next((c for c in CANDIDATES if os.path.exists(f'{c}/bilingual_test.csv')), None)

def load_probs(path):
    q =np.load(path).astype(np.float64)
    if q[:, 1].max() < 0.75:
        p1=np.clip((np.log(q[:, 1] / q[:, 0]) + 1) / 2, 0, 1)
        return np.stack([1 - p1, p1], 1)
    return q

FALLBACK = {'distilbert_en_only': 0.9262}
def f1_of(tag):
    path =f'{SRC}/{tag}_metrics.json'
    if os.path.exists(path):
        with open(path) as fh:
            return json.load(fh)['macro avg']['f1-score']
    if tag in FALLBACK:
        print(f"  [{tag}] metrics file absent, using retained notebook output 0.9262")
        return FALLBACK[tag]
    raise FileNotFoundError(path)

MODELS =['distilbert', 'mbert', 'xlmr']
DIRS=['en2es', 'es2en']
CEIL= {'en2es': '_es_only', 'es2en': '_en_only'}

In [ ]:
# 1: threshold sweep on the cross-lingual transfer conditions

rows=[]
runs= [(m, d) for m in MODELS for d in DIRS] + [('beto', 'es2en')]

for m, d in runs:
    p= load_probs(f'{SRC}/{m}_{d}_probs.npy')[:, 1]
    y=np.load(f'{SRC}/{m}_{d}_labels.npy')

    cand= np.unique(p)
    if len(cand)> 4000:
        cand=np.unique(np.quantile(p, np.linspace(0, 1, 4000)))

    scores =np.array([f1_score(y, (p >= t).astype(int), average='macro') for t in cand])
    best_i= scores.argmax()

    at_half= f1_score(y, (p >= 0.5).astype(int), average='macro')
    all_ab=f1_score(y, np.ones_like(y), average='macro')
    ceiling=f1_of('beto_es_only') if m == 'beto' else f1_of(f'{m}{CEIL[d]}')

    rows.append({
        'Model': m,
        'Direction': d,
        'F1 at 0.50': round(at_half, 4),
        'Best F1': round(scores[best_i], 4),
        'Best threshold': f'{cand[best_i]:.2e}',
        'F1 if all abusive': round(all_ab, 4),
        'Abusive pred at best (%)': round(100 * (p >= cand[best_i]).mean(), 1),
        'Gain (pp)': round(100 * (scores[best_i] - at_half), 2),
        'Ceiling': round(ceiling, 4),
        'Gap remaining (pp)': round(100 * (ceiling - scores[best_i]), 2),
        'Gap closed (%)': round(100 * (scores[best_i] - at_half) / (ceiling - at_half), 2),
    })

sweep_df= pd.DataFrame(rows)
sweep_df.to_csv(f'{OUT}/threshold_sweep.csv', index=False)
print(sweep_df.to_string(index=False))


  [distilbert_en_only] metrics file absent - using retained notebook output 0.9262
     Model Direction  F1 at 0.50  Best F1 Best threshold  F1 if all abusive  Abusive pred at best (%)  Gain (pp)  Ceiling  Gap remaining (pp)  Gap closed (%)
distilbert     en2es      0.4301   0.5878       6.32e-03             0.3333                      45.3      15.78   0.8684               28.05           35.99
distilbert     es2en      0.5396   0.6212       2.76e-02             0.3331                      54.3       8.16   0.9262               30.50           21.10
     mbert     en2es      0.4077   0.5442       2.45e-04             0.3333                      48.6      13.65   0.8773               33.31           29.07
     mbert     es2en      0.5456   0.6400       9.67e-03             0.3331                      55.7       9.45   0.9257               28.57           24.85
      xlmr     en2es      0.4942   0.5650       3.46e-02             0.3333                      44.6       7.08   0.8860      

In [ ]:
from transformers import AutoTokenizer

TOKENIZERS ={
    'distilbert': 'distilbert-base-multilingual-cased',
    'mbert':'bert-base-multilingual-cased',
    'xlmr':'xlm-roberta-base',
    'beto':'dccuchile/bert-base-spanish-wwm-cased',
}
TEXTS={
    'English':pd.read_csv(f'{SPLITS}/english_test.csv')['text'].astype(str).tolist(),
    'Spanish': pd.read_csv(f'{SPLITS}/spanish_test.csv')['text'].astype(str).tolist(),
}

rows =[]
for name, ckpt in TOKENIZERS.items():
    tok=AutoTokenizer.from_pretrained(ckpt)
    row= {'Model': name, 'Vocab size': tok.vocab_size}
    for lang, texts in TEXTS.items():
        n_sub =sum(len(tok.tokenize(t)) for t in texts)
        n_word= sum(len(t.split()) for t in texts)
        over =sum(len(tok.tokenize(t))> 128 for t in texts)
        row[f'{lang} tokens/word']=round(n_sub / n_word, 3)
        row[f'{lang} >128 (%)'] =round(100 * over / len(texts), 2)
    row['ES/EN fertility ratio']= round(row['Spanish tokens/word'] / row['English tokens/word'], 3)
    rows.append(row)

fert_df =pd.DataFrame(rows)
fert_df.to_csv(f'{OUT}/subword_fertility.csv', index=False)
print(fert_df.to_string(index=False))


config.json:   0%|          | 0.00/466 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (905 > 512). Running this sequence through the model will result in indexing errors


...

vocab.txt:   0%|          | 0.00/242k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/480k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1176 > 512). Running this sequence through the model will result in indexing errors


     Model  Vocab size  English tokens/word  English >128 (%)  Spanish tokens/word  Spanish >128 (%)  ES/EN fertility ratio
distilbert      119547                1.610              0.07                1.553              2.46                  0.965
     mbert      119547                1.610              0.07                1.553              2.46                  0.965
      xlmr      250002                1.566              0.03                1.456              2.05                  0.930
      beto       31002                2.178              0.10                1.378              1.84                  0.633





In [ ]:
es_test= pd.read_csv(f'{SPLITS}/spanish_test.csv')
y_es =np.load(f'{SRC}/beto_es_only_labels.npy')
assert np.array_equal(y_es, es_test.binary_label.values), "spanish_test.csv is not row-aligned"

es_test['beto_err'] =(np.load(f'{SRC}/beto_es_only_preds.npy') != y_es).astype(int)
es_test['xlmr_err']=(np.load(f'{SRC}/xlmr_es_only_preds.npy') != y_es).astype(int)

src =(es_test.groupby('source')
       .agg(n=('binary_label', 'size'),
            pct_abusive=('binary_label',lambda s: round(100 * s.mean(), 1)),
            beto=('beto_err', lambda s: round(100 * s.mean(), 2)),
            xlmr  =('xlmr_err', lambda s:round(100 * s.mean(), 2)))
       .reset_index())
src['BETO - XLM-R (pp)']= (src['beto'] - src['xlmr']).round(2)
src=src.sort_values('n', ascending=False)
src.to_csv(f'{OUT}/spanish_source_errors.csv', index=False)
print(src.to_string(index=False))
print(f"\noverall  BETO {100*es_test.beto_err.mean():.2f}%   XLM-R {100*es_test.xlmr_err.mean():.2f}%")


       source    n  pct_abusive  beto  xlmr  BETO - XLM-R (pp)
     OffendES 2668         27.2 11.66 12.78              -1.12
    Colombian  398         67.3 20.35 19.60               0.75
   ES_hateval  397        100.0  1.76  4.28              -2.52
  ES_haternet  258        100.0  2.33  4.26              -1.93
ES_misocorpus  214        100.0  1.40  3.27              -1.87
   ES_chileno  102        100.0  2.94  9.80              -6.86
  ES_hascosva   64        100.0  3.12  3.12               0.00
   ES_homomex   40        100.0 12.50 15.00              -2.50

overall  BETO 10.09%   XLM-R 11.40%



In [ ]:
profanity_en ={'fuck','fucking','fucked','shit','bitch','bastard','cunt','asshole','ass','dick', 'whore','slut','retard','retarded','faggot','nigger','idiot','stupid','moron','dumb','ugly',
    'fat','loser','kill','die','hate'}
profanity_es = {'puta','puto','putas','putos','mierda','joder','cabron','cabrón','gilipollas','pendejo','pendeja','imbecil','imbécil','idiota','estupido','estúpido','tonto','zorra','perra',
    'maricon','maricón','coño','carajo','muere','odio','asco','feo'}

def has_profanity(text, lexicon):
    return len(set(re.findall(r'\b\w+\b',str(text).lower())) & lexicon)>0

test =pd.read_csv(f'{SPLITS}/bilingual_test.csv')
y=np.load(f'{SRC}/ensemble_bilingual_full_labels.npy')
assert np.array_equal(y, test.binary_label.values), "bilingual_test.csv is not row-aligned"
test['err'] =(np.load(f'{SRC}/ensemble_bilingual_full_preds.npy') != y).astype(int)
test['profane']= test.apply(
    lambda r: has_profanity(r['text'], profanity_en if r['language'] == 'en' else profanity_es), axis=1)
test['harm_type'] = np.where(test['profane'], 'Explicit', 'Implicit')

ab =test[test.binary_label == 1]
print(f"abusive test rows: {len(ab)}   implicit:{100*(ab.harm_type=='Implicit').mean():.1f}%")

imp=(ab.groupby(['language', 'harm_type'])
       .agg(n=('err', 'size'), error_rate=('err', lambda s: round(100 * s.mean(), 2)))
       .reset_index())
imp.to_csv(f'{OUT}/implicit_vs_explicit_errors.csv',index=False)
print(imp.to_string(index=False))

for lg in ['en','es']:
    d =imp[imp.language == lg].set_index('harm_type')['error_rate']
    print(f"{lg}: implicit {d.get('Implicit')}%  explicit {d.get('Explicit')}%"
          f" difference {d.get('Implicit', 0) - d.get('Explicit', 0):+.2f} pp")

abusive test rows: 6987   implicit: 66.6%   (corpus-wide figure in Section 3.4 is 65.8%)

language harm_type    n  error_rate
      en  Explicit 1643        0.85
      en  Implicit 3274        8.89
      es  Explicit  694        6.48
      es  Implicit 1376       11.41
  en: implicit 8.89%  explicit 0.85%  difference +8.04 pp
  es: implicit 11.41%  explicit 6.48%  difference +4.93 pp


In [ ]:
for d in DIRS:
    y =np.load(f'{SRC}/{MODELS[0]}_{d}_labels.npy')
    err= {m: (np.load(f'{SRC}/{m}_{d}_preds.npy') != y) for m in MODELS}
    E+np.stack([err[m] for m in MODELS])

    print(f"\n{d}  (n={len(y)})")
    print("pairwise Jaccard overlap of error sets")
    for a, b in itertools.combinations(MODELS, 2):
        inter, union = (err[a] & err[b]).sum(), (err[a] | err[b]).sum()
        print(f"  {a:11s} & {b:11s}  {inter/union:.3f}   ({inter} shared of {union})")

    n_wrong =E.sum(0)
    dist=pd.Series(n_wrong).value_counts().sort_index()
    print("rows by number of models failing:")
    for k, v in dist.items():
        print(f"  {k} model(s) wrong: {v:6d}  ({100*v/len(y):5.2f}%)")

    allw =(n_wrong == 3)
    print(f"all three wrong on {allw.sum()} rows; of those, "
          f"{(y[allw] == 1).sum()} are abusive ({100*(y[allw]==1).mean():.1f}%)")


=== en2es  (n=4141) ===
pairwise Jaccard overlap of error sets
  distilbert  & mbert        0.878   (1871 shared of 2130)
  distilbert  & xlmr         0.675   (1585 shared of 2349)
  mbert       & xlmr         0.671   (1606 shared of 2395)
rows by number of models failing:
  0 model(s) wrong:   1704  (41.15%)
  1 model(s) wrong:    437  (10.55%)
  2 model(s) wrong:    469  (11.33%)
  3 model(s) wrong:   1531  (36.97%)
  all three wrong on 1531 rows; of those, 1447 are abusive (94.5%)

=== es2en  (n=9843) ===
pairwise Jaccard overlap of error sets
  distilbert  & mbert        0.788   (3595 shared of 4563)
  distilbert  & xlmr         0.628   (2912 shared of 4636)
  mbert       & xlmr         0.661   (2974 shared of 4502)
rows by number of models failing:
  0 model(s) wrong:   4983  (50.62%)
  1 model(s) wrong:    879  ( 8.93%)
  2 model(s) wrong:   1231  (12.51%)
  3 model(s) wrong:   2750  (27.94%)
  all three wrong on 2750 rows; of those, 2490 are abusive (90.5%)


In [ ]:
#6: what the three transformers miss together, in-language
import re

test =pd.read_csv(f'{SPLITS}/bilingual_test.csv')
y =np.load(f'{SRC}/xlmr_bilingual_full_labels.npy')
assert np.array_equal(y, test.binary_label.values), "bilingual_test.csv is not row-aligned"

TRIO= ['distilbert', 'mbert', 'xlmr']
wrong= np.stack([np.load(f'{SRC}/{m}_bilingual_full_preds.npy') != y for m in TRIO])
all_wrong=wrong.all(0)

print(f"rows all three miss: {all_wrong.sum()}  ({100*all_wrong.mean():.2f}% of {len(y)})")
print(f"abusive: {(y[all_wrong]==1).sum()}  ({100*(y[all_wrong]==1).mean():.1f}%)")
print(f" test set is {100*(y==1).mean():.1f}% abusive overall\n")

profanity_en = {'fuck','fucking','fucked','shit','bitch','bastard','cunt','asshole','ass','dick',
    'whore','slut','retard','retarded','faggot','nigger','idiot','stupid','moron','dumb','ugly',
    'fat','loser','kill','die','hate'}
profanity_es = {'puta','puto','putas','putos','mierda','joder','cabron','cabrón','gilipollas',
    'pendejo','pendeja','imbecil','imbécil','idiota','estupido','estúpido','tonto','zorra','perra',
    'maricon','maricón','coño','carajo','muere','odio','asco','feo'}

def implicit(r):
    lex = profanity_en if r['language'] == 'en' else profanity_es
    return not (set(re.findall(r'\b\w+\b', str(r['text']).lower())) & lex)

test['all_wrong'] = all_wrong
test['implicit']=test.apply(implicit, axis=1)
ab = test[test.binary_label == 1]

miss = ab[ab.all_wrong]
print(f"abusive rows all three miss: {len(miss)}")
print(f" implicit (no profanity): {miss.implicit.sum()}  ({100*miss.implicit.mean():.1f}%)")
print(f"baseline: {100*ab.implicit.mean():.1f}% of all abusive test rows are implicit\n")

print("error rate on abusive rows, by expression:")
for lab, grp in ab.groupby('implicit'):
    name = 'implicit' if lab else 'explicit'
    print(f"{name:9s} n={len(grp):5d}  all three wrong on {grp.all_wrong.sum():4d} "
          f"({100*grp.all_wrong.mean():.2f}%)")

test[['text','language','source','binary_label','implicit']][test.all_wrong] \
    .to_csv(f'{OUT}/all_three_wrong.csv', index=False)

rows all three miss: 664  (4.75% of 13984)
  abusive: 260  (39.2%)
  test set is 50.0% abusive overall

abusive rows all three miss: 260
  implicit (no profanity): 232  (89.2%)
  baseline: 66.6% of all abusive test rows are implicit

error rate on abusive rows, by expression:
  explicit  n= 2337  all three wrong on   28 (1.20%)
  implicit  n= 4650  all three wrong on  232 (4.99%)


In [ ]:
na =test[(test.binary_label == 0)]
print("non-abusive rows all three miss:", na.all_wrong.sum())
print("  containing profanity:", (~na[na.all_wrong].implicit).sum(),
      f"({100*(~na[na.all_wrong].implicit).mean():.1f}%)",
      f"vs {100*(~na.implicit).mean():.1f}% baseline")

non-abusive rows all three miss: 404
  containing profanity: 75 (18.6%) vs 5.0% baseline
